<div style="background: linear-gradient(120deg, #1a3a5c 0%, #2d6a9f 60%, #4a9eda 100%); padding: 28px 36px; border-radius: 14px; display: flex; align-items: center; gap: 28px; box-shadow: 0 4px 18px rgba(0,0,0,0.18);">
    <img src='Figures/iteso.jpg' style="height: 110px; border-radius: 8px; background: white; padding: 6px; flex-shrink: 0; box-shadow: 0 2px 8px rgba(0,0,0,0.2);"/>
    <div style="border-left: 2px solid rgba(255,255,255,0.4); padding-left: 28px;">
        <h1 style="margin: 0 0 8px 0; color: white; font-size: 1.5em; line-height: 1.3;">Transformaciones por Potencias</h1>
        <h3 style="margin: 0 0 8px 0; color: white; font-size: 1.15em; line-height: 1.3;">Ingeniería de Características</h3>
        <h3 style="margin: 0; color: rgba(255,255,255,0.8); font-weight: normal; font-size: 1.05em;">Maestría en Ciencia de Datos</h3>
    </div>
</div>

Cuando el sesgo de la distribución es negativo (cola más larga hacia la izquierda) nos interesará lo contrario: comprimir la cola de la izquierda y extender la de la derecha. Si lo pensamos, esto puede hacerse elevando cada valor al cuadrado o al cubo. Los productos resultantes de los valores pequeños estarán menos alejados que los resultantes de valores grandes, con lo que la distribución se parecerá más a una normal.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

df=pd.read_csv('numericos.csv')
df.head()

Aplicando una transformación de la forma
$$
x'=x^a,
$$
donde $x\in[1,\infty)$, obtuvimos los siguientes exponentes para cada variable

Variable|Sesgo original|$a$
--|--|--
'0'|6.0|-11
'1'|-2.77|17
'2'|-1.01|3

In [ ]:
#Sesgo
df.skew()

In [ ]:
df.hist();

In [ ]:
df_copy = df.copy()

In [ ]:
# Compensando la variable '0'
# Variable desplazada a 1
x = df['0'] - df['0'].min() + 1

In [ ]:
#Aplicando la transformación de potencias con a=-11
a = -11
df_copy['0'] = x**a
df_copy['0'].skew()

In [ ]:
df_copy['0'].hist()

In [ ]:
#Aplicando la transformación de potencias con a=17


In [ ]:
# Compensando la variable '1'
a = 17
x = df['1'] - df['1'].min() + 1
df_copy['1'] = x**a
df_copy['1'].skew()

In [ ]:
#Aplicando la transformación de potencias con a=3


In [ ]:
# Compensando la variable '2'
a = 3
x = df['2'] - df['2'].min() + 1
df_copy['2'] = x**a
df_copy['2'].skew()

In [ ]:
df_copy.hist();

---

# Familia de transformaciones de potencia
$$
x'=\frac{x^\lambda-1}{\lambda},
$$
haciendo $\displaystyle\lim_{\lambda\rightarrow0}\frac{x^\lambda-1}{\lambda}$ se tiene
$$
x'=\ln(x).
$$

## Transformación Box-Cox
$$
x'=\left\{\begin{array}{lr}
\dfrac{x^\lambda-1}{\lambda} & \lambda\neq0\\
\ln(x) & \lambda=0
\end{array}\right.
$$

In [ ]:
LMB = [-5,-1,0.5,0,2,3,4]
LMB


In [ ]:
LMB = [-5,-1,0.5,0,2,3,4]
x = np.linspace(0.1,2,100)
x

In [ ]:
def potencia(x,lmb):
    if lmb==0:
        return np.log(x)
    else:
        return (x**lmb -1)/lmb

In [ ]:
for lmb in LMB:
    xp = potencia(x,lmb)
    plt.plot(x, xp, label='$x^{('+str(lmb)+')}$')
plt.plot(x,x-1, '--k')
plt.legend()
plt.ylim([-2,2])
plt.grid()

#### Transformación Box-Cox Librerías

In [ ]:
from scipy.stats import boxcox

In [ ]:
sesgo=pd.read_csv('sesgo.csv')
sesgo.head(2)

In [ ]:
sesgo.shape

In [ ]:
#Mejor exponente mejor sesgo
LMB = np.linspace(-20,20,100)
LMB

In [ ]:
pd.DataFrame(boxcox(sesgo['1.0'],-1)).skew()[0]

In [ ]:
mejor_exp = {}
for v in sesgo:
    x = sesgo[v] - sesgo[v].min() + 1 #Variable desplazada
    sp = [pd.DataFrame(boxcox(x,lmb)).skew().values[0] for lmb in LMB]
    indx = np.argmin(np.abs(sp))
    mejor_exp[v]=(LMB[indx], sp[indx])
mejor_exp

In [ ]:
#hacer la función boxcox_min
def boxcox_min(LMB, df):
    mejor_exp = {}
    for v in sesgo:
        x = sesgo[v] - sesgo[v].min() + 1 #Variable desplazada
        sp = [pd.DataFrame(boxcox(x,lmb)).skew().values[0] for lmb in LMB]
        indx = np.argmin(np.abs(sp))
        mejor_exp[v]=(LMB[indx], sp[indx])
    return mejor_exp
    

In [ ]:
#Probar
boxcox_min(LMB, df)

In [ ]:
help(boxcox)

In [ ]:
# Obtener el mejor sesgo y lambda 
mejor_exp ={}
for v in sesgo:
    x = sesgo[v] - sesgo[v].min() + 1
    xp, lmb = boxcox(x)
    mejor_exp[v] = (lmb, pd.DataFrame(xp).skew().values[0])
mejor_exp

In [ ]:
# De la Columna '-0.5' obtener x, l
mejor_exp['-0.5']

In [ ]:
x, l = boxcox(sesgo['-0.5'])
l

In [ ]:
#Mediante la funcion boxcox


In [ ]:
#Valores de lambda


---
Uso de valores negativos para $x$:
$$
x'=\left\{
  \begin{array}{ll}
  \mathrm{sign}(x)\dfrac{(|x|+1)^\lambda-1}{\lambda},&\lambda\neq0\\
  \mathrm{sign}(x)\ln(|x|+1),&\lambda=0
  \end{array}.
  \right.
$$

In [ ]:
LMB = [-1,0.5,0,2,3,4]
x = np.linspace(-10,10,500)

In [ ]:
#Potencias para valores negativos, función
def potenciasSigno(x, lmb):
    if lmb==0:
        return np.sign(x)*np.log(abs(x) + 1)
    else:
        return np.sign(x)*((np.abs(x) + 1)**lmb - 1)/lmb

    

In [ ]:
# Visualización de la función para valores positivos y negativos
for lmb in LMB:
    xp = potenciasSigno(x,lmb)
    plt.plot(x, xp, label=f'exp={lmb}')
plt.plot(x,np.sign(x)*np.abs(x), '--k')
plt.legend()
plt.ylim([-10,10])
plt.grid()

## Yeo-Johnson
$$
x'=\left\{
  \begin{array}{lr}
  \frac{(x+1)^{\lambda_1}-1}{\lambda_1} & \lambda_1\neq0,\quad x\geq0\\
  \ln(x+1) & \lambda_1=0,\quad x\geq0\\
  -\frac{(-x+1)^{\lambda_2}-1}{\lambda_2} & \lambda_2\neq0,\quad x<0\\
  -\ln(-x+1) & \lambda_2=0,\quad x<0.
  \end{array}
  \right.
$$
Donde $\lambda_1+\lambda_2=2$.

Se sustituye $\lambda_2=2-\lambda_1$.
$$
x'=\left\{
  \begin{array}{lr}
  \frac{(x+1)^{\lambda}-1}{\lambda} & \lambda\neq0,\quad x\geq0\\
  \ln(x+1) & \lambda=0,\quad x\geq0\\
  -\frac{(-x+1)^{2-\lambda}-1}{2-\lambda} & \lambda\neq2,\quad x<0\\
  -\ln(-x+1) & \lambda=2,\quad x<0.
  \end{array}
  \right.
$$

In [ ]:
#Librerías con scipy
from scipy.stats import yeojohnson

In [ ]:
LMB

In [ ]:
#Visualización de los datos con yeojohnson
for lmb in LMB:
    xp = yeojohnson(x,lmb)
    plt.plot(x, xp, label=f'exp={lmb}')
plt.plot(x,np.sign(x)*np.abs(x), '--k')
plt.legend()
plt.ylim([-10,10])
plt.grid()

In [ ]:
#Histogama de dataframe df
df.hist();

In [ ]:
#Copia de df
# yeojohnson para la variable 0
# Sesgo resultante sin desplazar = 0.04

In [ ]:
# yeojohnson para la variable 1
df_copia = df.copy()

In [ ]:
df_copia['0'], l = yeojohnson(df['0'])
df_copia['0'].hist(bins=50)
l, df_copia['0'].skew()

In [ ]:
df_copia['1'], l = yeojohnson(df['1'])
df_copia['1'].hist(bins=50)
l, df_copia['1'].skew()

In [ ]:
# yeojohnson para la variable 2
df_copia['2'], l = yeojohnson(df['2'])
df_copia['2'].hist(bins=50)
l, df_copia['2'].skew()

---
## Ejemplo de Regresión Lineal

In [ ]:
reg = pd.read_csv('regresion.csv')

In [ ]:
reg.head()

In [ ]:
plt.scatter(reg['x'], reg['y'], s=5)
plt.grid()
plt.show()

In [ ]:
#Aplicar la transformación a y
reg.hist()

In [ ]:
reg['yp'],l = yeojohnson(reg['y'] )
l

In [ ]:
reg['yp'].hist()

In [ ]:
lin = LinearRegression()
lin.fit(reg[['x']], reg[['yp']])


In [ ]:
#Train test split


In [ ]:
#Visualización de yp


In [ ]:
x_fit = np.linspace(0,1.5,100).reshape(1,-1)

$$
x=\left\{
  \begin{array}{lr}
  \frac{(x'+1)^{\lambda_1}-1}{\lambda_1} & \lambda_1\neq0,\quad x'\geq0\\
  \ln(x'+1) & \lambda_1=0,\quad x\geq0\\
  -\frac{(-x+1)^{\lambda_2}-1}{\lambda_2} & \lambda_2\neq0,\quad x'<0\\
  -\ln(-x+1) & \lambda_2=0,\quad x'<0.
  \end{array}
  \right.
$$

In [ ]:
#Implementar la función inversa de yeojohnson


#### Ejemplo 1: Reducción de asimetría y mejora de normalidad

Usaremos la variable `MedInc` (ingreso medio) del California Housing para comparar la asimetría y normalidad antes y después de aplicar Box-Cox y Yeo-Johnson.

In [ ]:
from scipy.stats import boxcox, skew, normaltest
from sklearn.preprocessing import PowerTransformer
from sklearn.datasets import fetch_california_housing

california = fetch_california_housing()
df = pd.DataFrame(california.data, columns=california.feature_names)

x = df['MedInc']

# Box-Cox (solo positivos)
x_boxcox, lmbda = boxcox(x)

# Yeo-Johnson (acepta negativos, pero aquí todos son positivos)
pt = PowerTransformer(method='yeo-johnson')
x_yj = pt.fit_transform(x.values.reshape(-1,1)).flatten()


In [ ]:

print('Asimetría original:', skew(x))
print('Asimetría Box-Cox:', skew(x_boxcox))
print('Asimetría Yeo-Johnson:', skew(x_yj))

print('Normalidad original:', normaltest(x).pvalue)
print('Normalidad Box-Cox:', normaltest(x_boxcox).pvalue)
print('Normalidad Yeo-Johnson:', normaltest(x_yj).pvalue)

plt.figure(figsize=(15,4))
plt.subplot(1,3,1); plt.hist(x, bins=30); plt.title('Original')
plt.subplot(1,3,2); plt.hist(x_boxcox, bins=30); plt.title('Box-Cox')
plt.subplot(1,3,3); plt.hist(x_yj, bins=30); plt.title('Yeo-Johnson')
plt.show()

#### Ejemplo 2: Mejorar el desempeño de un modelo lineal

Compara el desempeño de una regresión lineal simple para predecir el valor medio de la casa (`MedHouseVal`) usando la variable `AveOccup` original y transformada (Box-Cox y Yeo-Johnson).


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

X_orig = df[['AveOccup']]
y = fetch_california_housing().target

# Box-Cox (solo positivos)
X_boxcox, _ = boxcox(df['AveOccup'] + 1)

# Yeo-Johnson
pt = PowerTransformer(method='yeo-johnson')
X_yj = pt.fit_transform(df[['AveOccup']])


In [ ]:
# Modelos
lr_orig = LinearRegression().fit(X_orig, y)
lr_boxcox = LinearRegression().fit(X_boxcox.reshape(-1,1), y)
lr_yj = LinearRegression().fit(X_yj, y)

y_pred_orig = lr_orig.predict(X_orig)
y_pred_boxcox = lr_boxcox.predict(X_boxcox.reshape(-1,1))
y_pred_yj = lr_yj.predict(X_yj)

In [ ]:
print('MSE original:', mean_squared_error(y, y_pred_orig))
print('MSE Box-Cox:', mean_squared_error(y, y_pred_boxcox))
print('MSE Yeo-Johnson:', mean_squared_error(y, y_pred_yj))

#### Ejemplo3: Mejorando modelos multivariados con transformaciones

En este ejemplo, se comparan modelos de regresión lineal multivariada usando variables originales y transformadas (Box-Cox y Yeo-Johnson) para predecir el valor medio de la casa (`MedHouseVal`).

In [ ]:
from sklearn.model_selection import train_test_split

# Seleccionamos varias variables numéricas
X = df[['MedInc', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup']]
y = fetch_california_housing().target


In [ ]:
# Box-Cox requiere todas las variables positivas
X_boxcox = X.copy()
for col in X_boxcox.columns:
    X_boxcox[col], _ = boxcox(X_boxcox[col] + 1)

# Yeo-Johnson acepta negativos
pt_multi = PowerTransformer(method='yeo-johnson')
X_yj = pt_multi.fit_transform(X)


In [ ]:
# División train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)
Xb_train, Xb_test, _, _ = train_test_split(X_boxcox, y, random_state=42)
Xy_train, Xy_test, _, _ = train_test_split(X_yj, y, random_state=42)


In [ ]:
# Modelos multivariados
lr_orig = LinearRegression().fit(X_train, y_train)
lr_boxcox = LinearRegression().fit(Xb_train, y_train)
lr_yj = LinearRegression().fit(Xy_train, y_train)


In [ ]:
y_pred_orig = lr_orig.predict(X_test)
y_pred_boxcox = lr_boxcox.predict(Xb_test)
y_pred_yj = lr_yj.predict(Xy_test)

print('MSE multivariado original:', mean_squared_error(y_test, y_pred_orig))
print('MSE multivariado Box-Cox:', mean_squared_error(y_test, y_pred_boxcox))
print('MSE multivariado Yeo-Johnson:', mean_squared_error(y_test, y_pred_yj))

Estas transformaciones no solo mejoran la simetría, sino que también pueden tener beneficios adicionales, como facilitar la interpretación de los resultados y hacer que los datos sean más apropiados para ciertos análisis estadísticos.

Es importante señalar que la elección de la transformación adecuada depende de la naturaleza de los datos y los objetivos del análisis. La visualización de las distribuciones transformadas mediante histogramas proporciona una herramienta valiosa para evaluar la efectividad de las transformaciones en la mejora de la simetría.

- **Cuadrática y cúbica:** Útiles para modelar relaciones no lineales, pero pueden aumentar la asimetría.
- **Box-Cox:** Busca normalidad, solo para valores positivos.
- **Yeo-Johnson:** Busca normalidad, acepta valores negativos y ceros.


| Transformación | Ventajas | Desventajas | Cuándo usar |
|---------------|----------|-------------|-------------|
| Cuadrática/Cúbica | Modelar relaciones no lineales | Puede aumentar asimetría, sensible a outliers | Cuando se sospecha relación polinómica |
| Box-Cox | Normaliza, ajusta asimetría positiva | Solo positivos | Cuando la variable es positiva y asimétrica |
| Yeo-Johnson | Normaliza, acepta negativos y ceros | Menos interpretable, más costosa | Cuando hay valores negativos o ceros |

- Analizar la distribución antes y después de transformar.
- Usar Box-Cox o Yeo-Johnson si se busca la normalidad para modelos lineales o tests estadísticos.
- Usar potencias si se busca modelar relaciones polinómicas.